# PHEME Four Behavioral Signatures Experiment

原始单文件脚本被拆分为：

- `pheme_experiment_utils.py`：全部工具函数、数据加载、Prompt、实验运行与分析函数。
- `pheme_four_effects.ipynb`：实验配置、数据初始化以及四个独立实验。

四个实验各自位于一个独立代码单元中：

1. Threshold / Binary Activation
2. Anchoring
3. Format Sensitivity
4. Label Override

thread id:
THREAD_ID = "524947030616313856"
THREAD_ID = "524947716393414656"
THREAD_ID = "525046443103354880"


## 1. 导入工具模块与配置参数

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import os
import sys
import time

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "experiment_util.py").exists() and (REPO_ROOT / "Pheme" / "experiment_util.py").exists():
    REPO_ROOT = REPO_ROOT / "Pheme"
DATA_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import experiment_util as exp


# =========================
# Data configuration
# =========================
EVENT_DIR = DATA_ROOT / "ottawashooting"
THREAD_ID = "525046443103354880"
MIN_REACTIONS = 10
MAX_AGENTS = 200
OUTPUT_DIR = DATA_ROOT / "pheme_four_effects_results"
LLM_OUTPUT_DIR = DATA_ROOT / "output"
STANCE_DIR = DATA_ROOT / "stance"

# =========================
# Model configuration
# =========================
API_URL = "https://api.deepseek.com/chat/completions"
API_KEY = os.getenv("DEEPSEEK_API_KEY", os.getenv("API_KEY", ""))
MODEL = "deepseek-v4-pro"
INIT_MODEL = "deepseek-v4-pro"
USE_LLM_INIT = True

# =========================
# Runtime configuration
# =========================
TEMPERATURE = 0.7
REPETITIONS = 3
MAX_WORKERS = 5
SLEEP_BETWEEN = 0.2
MAX_NEIGHBORS_IN_PROMPT = 8
SEED = 59
TARGET_AGENTS = 30

# =========================
# Experiment controls
# =========================
THRESHOLD_TARGET_SCORES = [2]
THRESHOLD_COUNTS = [0, 1,2, 3,4, 5, 6,7,8]
THRESHOLD_TOTAL_NEIGHBORS = 8
THRESHOLD_SUPPORT_NEIGHBORS = THRESHOLD_TOTAL_NEIGHBORS
THRESHOLD_BASELINE_ENABLED = True

ANCHOR_TARGET_SCORES = [1, 2, 4, 5]
ANCHOR_SUPPORT_NEIGHBORS = 2
ANCHOR_OPPOSITE_NEIGHBORS = 2

FORMAT_TARGET_SCORES = [2]
FORMAT_SUPPORTS = 3
FORMAT_DENIES = 0

ROLE_FRAC = 0.2

if not API_KEY:
    print("Warning: DEEPSEEK_API_KEY is empty. Set DEEPSEEK_API_KEY before executing API cells.")

exp.configure_runtime(
    api_url=API_URL,
    api_key=API_KEY,
    model=MODEL,
    init_model=INIT_MODEL,
    max_workers=MAX_WORKERS,
    sleep_between=SLEEP_BETWEEN,
    max_neighbors_in_prompt=MAX_NEIGHBORS_IN_PROMPT,
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LLM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STANCE_DIR.mkdir(parents=True, exist_ok=True)


## 2. 加载 PHEME thread 并初始化实验状态

In [2]:
(
    selected_thread,
    GRAPH,
    AGENT_NAMES,
    AGENT_TEXTS,
    INITIAL_OPINIONS,
    TOPIC,
) = exp.load_and_set_pheme_thread(
    EVENT_DIR,
    thread_id=THREAD_ID,
    min_reactions=MIN_REACTIONS,
    max_agents=MAX_AGENTS,
    use_llm_init=USE_LLM_INIT,
    stance_dir=STANCE_DIR,
)

hub_nodes, peripheral_nodes, degrees = exp.build_role_groups(
    GRAPH,
    frac=ROLE_FRAC,
)

initial_distribution = dict(
    sorted(
        Counter(
            INITIAL_OPINIONS.values()
        ).items()
    )
)

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

all_results = {}
all_analyses = {}
pairwise = {}

print(f"Selected thread: {selected_thread}")
print(f"Topic: {TOPIC[:160]}")
print(f"Model: {MODEL}")
print(
    "Init model:",
    INIT_MODEL if USE_LLM_INIT else "heuristic",
)
print(f"Repetitions: {REPETITIONS}")

exp.summarize_graph(
    GRAPH,
    degrees,
)

print(
    "Initial stance distribution:",
    initial_distribution,
)

[PHEME] Selected specified thread: C:\Users\LiMingzhe\Desktop\LLM Simulation\test\ottawashooting\rumours\525046443103354880
[PHEME] Label folder: rumours
[PHEME] Reactions: 56
[Stance Cache] Loaded initial stances: C:\Users\LiMingzhe\Desktop\LLM Simulation\test\stance\525046443103354880_deepseek-v4-pro.json
[PHEME] structure.json not found. Built edges from reply metadata: 56
[Init] Agent 1/57
[Init Opinion] LLM not called | reason=stance_cache | score=4 | agent=0
[Init] Agent 2/57
[Init Opinion] LLM not called | reason=stance_cache | score=3 | agent=1
[Init] Agent 3/57
[Init Opinion] LLM not called | reason=stance_cache | score=3 | agent=2
[Init] Agent 4/57
[Init Opinion] LLM not called | reason=stance_cache | score=2 | agent=3
[Init] Agent 5/57
[Init Opinion] LLM not called | reason=stance_cache | score=4 | agent=4
[Init] Agent 6/57
[Init Opinion] LLM not called | reason=stance_cache | score=2 | agent=5
[Init] Agent 7/57
[Init Opinion] LLM not called | reason=stance_cache | score=2 |

## 3. Experiment 1 — Threshold / Binary Activation

同一批 Agent 分别看到 0、1、3、5、7、10 个反对型邻居，总邻居数固定为 10，比较意见变化率、向反对方向移动率和平均变化幅度。


In [3]:
# ============================================================
# Experiment 1:
# Threshold effect on the real comment-reply graph
# ============================================================
import experiment_util as exp

required_neighbor_count = max(
    THRESHOLD_COUNTS
)

# source tweet 节点通常是 0。
# source 可以作为邻居，但不作为目标 Agent。
target_comment_agents = sorted(
    agent_id
    for agent_id in INITIAL_OPINIONS
    if agent_id != 0
)

(
    threshold_neighbor_rankings,
    dropped_threshold_agents,
) = exp.build_threshold_neighbor_rankings(
    GRAPH,
    INITIAL_OPINIONS,
    target_agent_ids=target_comment_agents,
    required_count=required_neighbor_count,
    required_support_count=THRESHOLD_SUPPORT_NEIGHBORS,
    seed=SEED,
)

# T0、T1、T3、T5、T7、T10 使用相同的一批 Agent。
threshold_agents = sorted(
    threshold_neighbor_rankings
)

print(
    f"[Threshold] Candidate agents: "
    f"{len(target_comment_agents)}"
)

print(
    f"[Threshold] Valid agents: "
    f"{len(threshold_agents)}"
)

print(
    f"[Threshold] Dropped agents: "
    f"{len(dropped_threshold_agents)}"
)

if not threshold_agents:
    raise RuntimeError(
        "No comment agent has enough eligible "
        f"opposing comments for k={required_neighbor_count} "
        "and supporting comments for T0."
    )

threshold_analyses = []

if THRESHOLD_BASELINE_ENABLED:
    baseline_condition = "B0"
    baseline_label = "B0: No Neighbor Baseline"

    print("\n" + "=" * 72)

    print(
        "Running threshold baseline: "
        f"{baseline_condition}"
    )

    print(
        "Opposing neighbors: 0 | "
        "Supporting neighbors: 0 | "
        f"Agents: {len(threshold_agents)} | "
        f"Repetitions: {REPETITIONS}"
    )

    print("=" * 72)

    baseline_results = (
        exp.run_comment_threshold_condition(
            condition_name=(
                "B0_no_neighbor_baseline"
            ),
            agent_ids=threshold_agents,
            opinions=INITIAL_OPINIONS,
            topic=TOPIC,
            agent_names=AGENT_NAMES,
            agent_texts=AGENT_TEXTS,
            neighbor_rankings=(
                threshold_neighbor_rankings
            ),
            neighbor_count=0,
            total_neighbor_count=0,
            repetitions=REPETITIONS,
            temperature=TEMPERATURE,
        )
    )

    baseline_analysis = (
        exp.analyze_user_threshold_condition(
            baseline_label,
            baseline_results,
        )
    )

    exp.print_analysis(baseline_analysis)

    print(
        "  Opposite shift:      "
        f"{baseline_analysis['opposite_shift_count']}/"
        f"{baseline_analysis['polarized_n']} "
        f"({baseline_analysis['opposite_shift_rate']:.3f})"
    )

    print(
        "  Opposite final side: "
        f"{baseline_analysis['opposite_final_count']}/"
        f"{baseline_analysis['polarized_n']} "
        f"({baseline_analysis['opposite_final_rate']:.3f})"
    )

    print(
        "  Neutral changed rate:"
        f" {baseline_analysis['neutral_agent_changed_rate']:.3f}"
    )

    print(
        "  Avg real neighbors: "
        f"{baseline_analysis['avg_real_neighbor_count']:.3f}"
    )

    print(
        "  Avg random fallback:"
        f" {baseline_analysis['avg_random_neighbor_count']:.3f}"
    )

    print(
        "  Avg opposite neighbors:"
        f" {baseline_analysis['avg_opposite_neighbor_count']:.3f}"
    )

    print(
        "  Avg support neighbors: "
        f"{baseline_analysis['avg_support_neighbor_count']:.3f}"
    )

    baseline_output_path = exp.build_llm_output_file_path(
        LLM_OUTPUT_DIR,
        selected_thread=selected_thread,
        model=MODEL,
        condition=baseline_condition,
    )

    exp.save_llm_comment_score_outputs(
        baseline_output_path,
        selected_thread=selected_thread,
        topic=TOPIC,
        model=MODEL,
        init_model=INIT_MODEL,
        condition=baseline_condition,
        results=baseline_results,
        config={
            "thread_id": THREAD_ID,
            "condition": baseline_condition,
            "opposing_neighbors": 0,
            "supporting_neighbors": 0,
            "total_neighbors": 0,
            "repetitions": REPETITIONS,
            "temperature": TEMPERATURE,
            "seed": SEED,
        },
        timestamp=timestamp,
    )

    exp.print_generated_comment_samples(
        baseline_results,
        limit=5,
    )

    all_results[baseline_condition] = baseline_results
    all_analyses[baseline_condition] = baseline_analysis

    threshold_analyses.append(
        (baseline_condition, baseline_analysis)
    )

    time.sleep(SLEEP_BETWEEN)


for condition_index, k in enumerate(
    THRESHOLD_COUNTS,
    start=1,
):
    support_count = max(
        0,
        THRESHOLD_TOTAL_NEIGHBORS - k,
    )

    print("\n" + "=" * 72)

    print(
        f"Running threshold condition "
        f"{condition_index}/{len(THRESHOLD_COUNTS)}: "
        f"T{k}"
    )

    print(
        f"Opposing neighbors: {k} | "
        f"Supporting neighbors: {support_count} | "
        f"Agents: {len(threshold_agents)} | "
        f"Repetitions: {REPETITIONS}"
    )

    print("=" * 72)

    results = (
        exp.run_comment_threshold_condition(
            condition_name=(
                f"T{k}_real_comment_graph"
            ),
            agent_ids=threshold_agents,
            opinions=INITIAL_OPINIONS,
            topic=TOPIC,
            agent_names=AGENT_NAMES,
            agent_texts=AGENT_TEXTS,
            neighbor_rankings=(
                threshold_neighbor_rankings
            ),
            neighbor_count=k,
            total_neighbor_count=THRESHOLD_TOTAL_NEIGHBORS,
            repetitions=REPETITIONS,
            temperature=TEMPERATURE,
        )
    )

    analysis = (
        exp.analyze_user_threshold_condition(
            f"T{k}: {k} Opposing Comment Neighbors",
            results,
        )
    )

    exp.print_analysis(analysis)

    print(
        "  Opposite shift:      "
        f"{analysis['opposite_shift_count']}/"
        f"{analysis['polarized_n']} "
        f"({analysis['opposite_shift_rate']:.3f})"
    )

    print(
        "  Opposite final side: "
        f"{analysis['opposite_final_count']}/"
        f"{analysis['polarized_n']} "
        f"({analysis['opposite_final_rate']:.3f})"
    )

    print(
        "  Neutral changed rate:"
        f" {analysis['neutral_agent_changed_rate']:.3f}"
    )

    print(
        "  Avg real neighbors: "
        f"{analysis['avg_real_neighbor_count']:.3f}"
    )

    print(
        "  Avg random fallback:"
        f" {analysis['avg_random_neighbor_count']:.3f}"
    )

    print(
        "  Avg opposite neighbors:"
        f" {analysis['avg_opposite_neighbor_count']:.3f}"
    )

    print(
        "  Avg support neighbors: "
        f"{analysis['avg_support_neighbor_count']:.3f}"
    )

    llm_output_path = exp.build_llm_output_file_path(
        LLM_OUTPUT_DIR,
        selected_thread=selected_thread,
        model=MODEL,
        condition=f"T{k}",
    )

    exp.save_llm_comment_score_outputs(
        llm_output_path,
        selected_thread=selected_thread,
        topic=TOPIC,
        model=MODEL,
        init_model=INIT_MODEL,
        condition=f"T{k}",
        results=results,
        config={
            "thread_id": THREAD_ID,
            "condition": f"T{k}",
            "opposing_neighbors": k,
            "supporting_neighbors": support_count,
            "total_neighbors": THRESHOLD_TOTAL_NEIGHBORS,
            "repetitions": REPETITIONS,
            "temperature": TEMPERATURE,
            "seed": SEED,
        },
        timestamp=timestamp,
    )

    exp.print_generated_comment_samples(
        results,
        limit=5,
    )

    all_results[f"T{k}"] = results
    all_analyses[f"T{k}"] = analysis

    threshold_analyses.append(
        (f"T{k}", analysis)
    )

    time.sleep(SLEEP_BETWEEN)


# ============================================================
# Threshold summary
# ============================================================

print("\n" + "=" * 90)
print("Threshold experiment summary")
print("=" * 90)

print(
    f"{'Condition':<12}"
    f"{'Valid':>8}"
    f"{'Changed':>10}"
    f"{'ChangeRate':>12}"
    f"{'OppShift':>12}"
    f"{'AvgChange':>12}"
)

print("-" * 90)

for condition_label, analysis in threshold_analyses:
    print(
        f"{condition_label:<12}"
        f"{analysis['n']:>8}"
        f"{analysis['changed']:>10}"
        f"{analysis['changed_rate']:>12.3f}"
        f"{analysis['opposite_shift_rate']:>12.3f}"
        f"{analysis['avg_abs_change']:>12.3f}"
    )


[Threshold] Candidate agents: 56
[Threshold] Valid agents: 56
[Threshold] Dropped agents: 0

Running threshold condition 1/9: T0
Opposing neighbors: 0 | Supporting neighbors: 8 | Agents: 56 | Repetitions: 3

[T0_real_comment_graph] Starting 168 tasks: 56 agents × 3 repetitions
[T0_real_comment_graph] Task progress: 168/168 (100.0%) | valid=168 | failed=0
[T0_real_comment_graph] Finished | tasks=168 | valid=168 | failed=0

Condition: T0: 0 Opposing Comment Neighbors
  Valid calls:          168
  Changed:              47/168 (0.280)
  Avg |change|:         0.554
  Avg signed change:    0.268
  Avg final score:      3.429
  Keep current rate:    0.720
  Neutral final rate:   0.185
  Support final rate:   0.417
  Deny final rate:      0.399
  Variance:             1.900
  Distribution:         {1: 1, 2: 66, 3: 31, 5: 70}
  Transition diagnostics:
    Adjacent changes:   14/47 (0.298)
    Large jumps >=2:    33/47 (0.702)
    Polarity flips:     21/47 (0.447)
    Transition counts:
      2-


Running threshold condition 4/9: T3
Opposing neighbors: 3 | Supporting neighbors: 5 | Agents: 56 | Repetitions: 3

[T3_real_comment_graph] Starting 168 tasks: 56 agents × 3 repetitions
[T3_real_comment_graph] Task progress: 168/168 (100.0%) | valid=168 | failed=0
[T3_real_comment_graph] Finished | tasks=168 | valid=168 | failed=0

Condition: T3: 3 Opposing Comment Neighbors
  Valid calls:          168
  Changed:              68/168 (0.405)
  Avg |change|:         0.667
  Avg signed change:    0.417
  Avg final score:      3.577
  Keep current rate:    0.595
  Neutral final rate:   0.101
  Support final rate:   0.494
  Deny final rate:      0.405
  Variance:             2.065
  Distribution:         {1: 1, 2: 67, 3: 17, 5: 83}
  Transition diagnostics:
    Adjacent changes:   35/68 (0.515)
    Large jumps >=2:    33/68 (0.485)
    Polarity flips:     14/68 (0.206)
    Transition counts:
      2->3: 7
      2->5: 11
      3->1: 1
      3->2: 13
      3->5: 18
      4->2: 3
      4->5: 1


Running threshold condition 7/9: T6
Opposing neighbors: 6 | Supporting neighbors: 2 | Agents: 56 | Repetitions: 3

[T6_real_comment_graph] Starting 168 tasks: 56 agents × 3 repetitions
[T6_real_comment_graph] Task progress: 168/168 (100.0%) | valid=168 | failed=0
[T6_real_comment_graph] Finished | tasks=168 | valid=168 | failed=0

Condition: T6: 6 Opposing Comment Neighbors
  Valid calls:          168
  Changed:              73/168 (0.435)
  Avg |change|:         0.690
  Avg signed change:    0.321
  Avg final score:      3.482
  Keep current rate:    0.565
  Neutral final rate:   0.095
  Support final rate:   0.470
  Deny final rate:      0.435
  Variance:             2.119
  Distribution:         {1: 3, 2: 70, 3: 16, 4: 1, 5: 78}
  Transition diagnostics:
    Adjacent changes:   39/73 (0.534)
    Large jumps >=2:    34/73 (0.466)
    Polarity flips:     13/73 (0.178)
    Transition counts:
      2->1: 1
      2->3: 9
      2->5: 8
      3->1: 2
      3->2: 14
      3->4: 1
      3->


Threshold experiment summary
Condition      Valid   Changed  ChangeRate    OppShift   AvgChange
------------------------------------------------------------------------------------------
T0              168        47       0.280       0.167       0.554
T1              168        66       0.393       0.183       0.732
T2              168        74       0.440       0.143       0.756
T3              168        68       0.405       0.111       0.667
T4              168        74       0.440       0.143       0.762
T5              168        75       0.446       0.103       0.714
T6              168        73       0.435       0.103       0.690
T7              168        74       0.440       0.119       0.702
T8              168        70       0.417       0.079       0.661


## 4. Experiment 2 — Anchoring

所有目标 Agent 都看到相同的 2 个支持邻居和 2 个否认邻居，测试其最终意见更倾向于回到中立，还是保持初始立场。


In [3]:
# Experiment 2: Anchoring

# Use only agents whose initial stance is 1/2/4/5.
# All target agents see the same fixed shared anchor comments.
ANCHOR_FIXED_NEIGHBOR_IDS = [3, 18, 16, 21]

anchor_neighbor_items = []
for neighbor_id in ANCHOR_FIXED_NEIGHBOR_IDS:
    score = INITIAL_OPINIONS[neighbor_id]
    if score in {4, 5}:
        relation = "support"
    elif score in {1, 2}:
        relation = "opposite"
    else:
        relation = "neutral"

    anchor_neighbor_items.append({
        "agent_id": neighbor_id,
        "source": "fixed_shared_anchor",
        "relation": relation,
    })

anchor_support_count = sum(
    item["relation"] == "support"
    for item in anchor_neighbor_items
)
anchor_opposite_count = sum(
    item["relation"] == "opposite"
    for item in anchor_neighbor_items
)
anchor_neutral_count = sum(
    item["relation"] == "neutral"
    for item in anchor_neighbor_items
)

anchor_neighbor_ids = {
    item["agent_id"]
    for item in anchor_neighbor_items
}

anchor_agents = sorted(
    agent_id
    for agent_id, score in INITIAL_OPINIONS.items()
    if (
        agent_id != 0
        and agent_id not in anchor_neighbor_ids
        and score in ANCHOR_TARGET_SCORES
    )
)

print(f"[Anchoring] Target agents: {anchor_agents}")
print(
    "[Anchoring] Shared anchor neighbors:",
    [
        {
            "agent_id": item["agent_id"],
            "relation": item["relation"],
            "score": INITIAL_OPINIONS[item["agent_id"]],
            "text": AGENT_TEXTS[item["agent_id"]],
        }
        for item in anchor_neighbor_items
    ],
)

results_anchor = exp.run_comment_anchor_condition(
    condition_name="T4c_fixed_shared_anchor_comment_then_classify",
    agent_ids=anchor_agents,
    opinions=INITIAL_OPINIONS,
    topic=TOPIC,
    agent_names=AGENT_NAMES,
    agent_texts=AGENT_TEXTS,
    neighbor_items=anchor_neighbor_items,
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)

analysis_anchor = exp.analyze_user_threshold_condition(
    "T4c: Fixed Shared Anchor Comments",
    results_anchor,
)
exp.print_analysis(analysis_anchor)

print(
    "  Opposite shift:      "
    f"{analysis_anchor['opposite_shift_count']}/"
    f"{analysis_anchor['polarized_n']} "
    f"({analysis_anchor['opposite_shift_rate']:.3f})"
)

print(
    "  Opposite final side: "
    f"{analysis_anchor['opposite_final_count']}/"
    f"{analysis_anchor['polarized_n']} "
    f"({analysis_anchor['opposite_final_rate']:.3f})"
)

print(
    "  Neutral changed rate:"
    f" {analysis_anchor['neutral_agent_changed_rate']:.3f}"
)

print(
    "  Avg opposite neighbors:"
    f" {analysis_anchor['avg_opposite_neighbor_count']:.3f}"
)

print(
    "  Avg support neighbors: "
    f"{analysis_anchor['avg_support_neighbor_count']:.3f}"
)

anchor_output_path = exp.build_llm_output_file_path(
    LLM_OUTPUT_DIR,
    selected_thread=selected_thread,
    model=MODEL,
    condition="T4c_anchor",
)

exp.save_llm_comment_score_outputs(
    anchor_output_path,
    selected_thread=selected_thread,
    topic=TOPIC,
    model=MODEL,
    init_model=INIT_MODEL,
    condition="T4c_anchor",
    results=results_anchor,
    config={
        "thread_id": THREAD_ID,
        "condition": "T4c_anchor",
        "target_scores": ANCHOR_TARGET_SCORES,
        "fixed_anchor_neighbor_ids": ANCHOR_FIXED_NEIGHBOR_IDS,
        "supporting_neighbors": anchor_support_count,
        "opposing_neighbors": anchor_opposite_count,
        "neutral_neighbors": anchor_neutral_count,
        "shared_anchor_neighbor_ids": sorted(anchor_neighbor_ids),
        "shared_anchor_sampling": "fixed",
        "repetitions": REPETITIONS,
        "temperature": TEMPERATURE,
    },
    timestamp=timestamp,
)

exp.print_generated_comment_samples(
    results_anchor,
    limit=5,
)

all_results["T4c_anchor"] = results_anchor
all_analyses["T4c_anchor"] = analysis_anchor

print(
    "\nInterpretation: "
    "high keep_current_rate supports anchoring; "
    "high neutral_final_rate supports averaging."
)


[Anchoring] Target agents: [4, 5, 6, 8, 9, 11, 12, 13, 14, 20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 64, 67, 68, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 103]
[Anchoring] Shared anchor neighbors: [{'agent_id': 3, 'relation': 'support', 'score': 4, 'text': '@chrissyteigen people only hear/read what they wanna hear/read. As a Canadian who has also lived in the States, I respect you and your tweet'}, {'agent_id': 18, 'relation': 'support', 'score': 5, 'text': '@chrissyteigen Its so bad, we have a 4+ casualty shooting EVERY SINGLE DAY here. Proof: http://t.co/VxyVaqP4IT'}, {'agent_id': 16, 'relation': 'opposite', 'score': 1, 'text': "@chrissyteigen sorry darlin, America doesn't have issues with gun control, we have issues with criminals who don't follow current laws."}, {'agent_id': 21, 'relation': 'opposite',

## 5. Experiment 3 — Format Sensitivity

使用完全相同的 Agent 与邻居信息，仅改变 Prompt A/B 的表达格式，并进行逐 Agent、逐 repetition 的配对比较。


In [ ]:
# Experiment 3: Format Sensitivity

# 保持原脚本当前行为：使用除 source 节点 0 之外的全部 Agent。
format_agents = sorted(
    agent_id
    for agent_id in INITIAL_OPINIONS
    if agent_id != 0
)
print(f"[Format] Target agents: {format_agents}")

format_lines = exp.make_controlled_neighbor_lines(
    n_support=FORMAT_SUPPORTS,
    n_deny=FORMAT_DENIES,
    n_neutral=0,
)

results_format_a = exp.run_controlled_condition(
    condition_name="C1_prompt_A_same_info",
    agent_ids=format_agents,
    opinions=INITIAL_OPINIONS,
    topic=TOPIC,
    neighbor_lines=format_lines,
    prompt_variant="A",
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)
analysis_format_a = exp.analyze_condition(
    "C1: Same Info + Prompt A",
    results_format_a,
    direction="up",
)
exp.print_analysis(analysis_format_a)

time.sleep(SLEEP_BETWEEN)

results_format_b = exp.run_controlled_condition(
    condition_name="C3_prompt_B_same_info",
    agent_ids=format_agents,
    opinions=INITIAL_OPINIONS,
    topic=TOPIC,
    neighbor_lines=format_lines,
    prompt_variant="B",
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)
analysis_format_b = exp.analyze_condition(
    "C3: Same Info + Prompt B",
    results_format_b,
    direction="up",
)
exp.print_analysis(analysis_format_b)

comparison_format = exp.compare_pairwise(
    "C1 Prompt A vs C3 Prompt B",
    results_format_a,
    results_format_b,
)
exp.print_pairwise(comparison_format)

all_results["C1_format_A"] = results_format_a
all_results["C3_format_B"] = results_format_b
all_analyses["C1_format_A"] = analysis_format_a
all_analyses["C3_format_B"] = analysis_format_b
pairwise["format_C1_vs_C3"] = comparison_format


## 6. Experiment 4 — Label Override

在真实 PHEME 图上选择外围节点，对比“不提供角色标签”和“错误标记为 central hub”两种条件。


In [ ]:
# Experiment 4: Label Override

label_agents = sorted(peripheral_nodes[:TARGET_AGENTS])
print(f"[Label Override] True peripheral nodes: {label_agents}")

results_label_base = exp.run_real_graph_condition(
    condition_name="B7_base_peripheral_no_label_real_graph",
    agent_ids=label_agents,
    opinions=INITIAL_OPINIONS,
    graph=GRAPH,
    topic=TOPIC,
    prompt_variant="A",
    role_labels=None,
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)
analysis_label_base = exp.analyze_condition(
    "B7-base: True Peripheral + No Label",
    results_label_base,
)
exp.print_analysis(analysis_label_base)

time.sleep(SLEEP_BETWEEN)

hub_mislabels = {
    agent_id: "central hub"
    for agent_id in label_agents
}
results_label_hub = exp.run_real_graph_condition(
    condition_name="B7_peripheral_mislabeled_as_hub_real_graph",
    agent_ids=label_agents,
    opinions=INITIAL_OPINIONS,
    graph=GRAPH,
    topic=TOPIC,
    prompt_variant="A",
    role_labels=hub_mislabels,
    repetitions=REPETITIONS,
    temperature=TEMPERATURE,
)
analysis_label_hub = exp.analyze_condition(
    "B7: True Peripheral Mislabeled as Central Hub",
    results_label_hub,
)
exp.print_analysis(analysis_label_hub)

comparison_label = exp.compare_pairwise(
    "B7-base No Label vs B7 Mislabel Hub",
    results_label_base,
    results_label_hub,
)
exp.print_pairwise(comparison_label)

all_results["B7_base"] = results_label_base
all_results["B7_mislabel_hub"] = results_label_hub
all_analyses["B7_base"] = analysis_label_base
all_analyses["B7_mislabel_hub"] = analysis_label_hub
pairwise["label_B7_base_vs_mislabel"] = comparison_label


## 7. 汇总并保存全部实验结果

In [ ]:
print("\n" + "#" * 96)
print("SUMMARY TABLE")
print("#" * 96)
print(
    f"{'Condition':<48} "
    f"{'Changed':>10} "
    f"{'AvgChg':>8} "
    f"{'Keep':>8} "
    f"{'Neutral':>8} "
    f"{'Support':>8}"
)
print("-" * 96)

for key, analysis in all_analyses.items():
    print(
        f"{analysis['name']:<48} "
        f"{analysis['changed']}/{analysis['n']:<6} "
        f"{analysis['avg_abs_change']:>8.3f} "
        f"{analysis['keep_rate']:>8.3f} "
        f"{analysis['neutral_rate']:>8.3f} "
        f"{analysis['support_rate']:>8.3f}"
    )

config = {
    "event_dir": str(EVENT_DIR),
    "thread_id": THREAD_ID,
    "min_reactions": MIN_REACTIONS,
    "max_agents": MAX_AGENTS,
    "stance_dir": str(STANCE_DIR),
    "llm_output_dir": str(LLM_OUTPUT_DIR),
    "model": MODEL,
    "init_model": INIT_MODEL,
    "use_llm_init": USE_LLM_INIT,
    "temperature": TEMPERATURE,
    "repetitions": REPETITIONS,
    "max_workers": MAX_WORKERS,
    "sleep_between": SLEEP_BETWEEN,
    "max_neighbors_in_prompt": MAX_NEIGHBORS_IN_PROMPT,
    "seed": SEED,
    "target_agents": TARGET_AGENTS,
    "threshold_target_scores": THRESHOLD_TARGET_SCORES,
    "threshold_counts": THRESHOLD_COUNTS,
    "threshold_total_neighbors": THRESHOLD_TOTAL_NEIGHBORS,
    "threshold_support_neighbors": THRESHOLD_SUPPORT_NEIGHBORS,
    "threshold_baseline_enabled": THRESHOLD_BASELINE_ENABLED,
    "anchor_target_scores": ANCHOR_TARGET_SCORES,
    "anchor_support_neighbors": ANCHOR_SUPPORT_NEIGHBORS,
    "anchor_opposite_neighbors": ANCHOR_OPPOSITE_NEIGHBORS,
    "format_target_scores": FORMAT_TARGET_SCORES,
    "format_supports": FORMAT_SUPPORTS,
    "format_denies": FORMAT_DENIES,
    "role_frac": ROLE_FRAC,
}

graph_summary = {
    "nodes": len(GRAPH),
    "edges": len(exp.get_undirected_edges(GRAPH)),
    "degrees": degrees,
    "initial_distribution": initial_distribution,
    "hub_nodes": hub_nodes,
    "peripheral_nodes": peripheral_nodes,
}

output_path = OUTPUT_DIR / f"four_effects_results_{timestamp}.json"
saved_path = exp.save_experiment_bundle(
    output_path,
    config=config,
    selected_thread=selected_thread,
    topic=TOPIC,
    graph_summary=graph_summary,
    analyses=all_analyses,
    pairwise=pairwise,
    results=all_results,
    timestamp=timestamp,
)

print(f"\nFull results saved to: {saved_path}")
print(f"Total API calls used: {exp.call_count}")
